#### Download all tables in gold layers to csv

#####1. Copy tables to temp dbfs

In [0]:
#Import library
import shutil
import os
import time
from datetime import datetime

In [0]:
schema_name = "gold_olist"
dbfs_temp = "dbfs:/tmp/gold_olist_tables"
csv_temp = "dbfs/tmp/gold_olist_csvs"

# Remove old dbfs temp, and recreate it
dbutils.fs.rm(dbfs_temp, recurse=True)
dbutils.fs.mkdirs(dbfs_temp)

# Remove old csv_temp, and recreate it
dbutils.fs.rm(csv_temp, recurse=True)
dbutils.fs.mkdirs(csv_temp)

In [0]:
tables = [t.name for t in spark.catalog.listTables(schema_name) if t.tableType != "TEMPORARY"]
print("Tables found:", tables)

In [0]:
#List out the files in the DBFS temp folder
dbutils.fs.ls(dbfs_temp)

In [0]:
#Write tables to csv files in dbfs_temp
for table in tables:
    df = spark.table(f"{schema_name}.{table}")
    folder = f"{dbfs_temp}/{table}"
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(folder)
    print(f"{table} exported to {folder}")

In [0]:
#List out the files in the DBFS temp folder
display(dbutils.fs.ls(f"{dbfs_temp}"))

In [0]:
#Copy csv files from dbfs_temp(csv file in each folder) to csv_temp
for file in dbutils.fs.ls(f"{dbfs_temp}"):
    for f in dbutils.fs.ls(file.path):
        if f.name.lower().endswith(".csv"):
            #Copy to csv_temp with right name file
            dbutils.fs.cp(f.path, f"{csv_temp}/{file.name.strip('/')}.csv")
#List out files in csv_temp
display(dbutils.fs.ls(csv_temp))           

##### 2. Zip csv files for download

In [0]:
#Make csv_temp_local to copy all files from csv_temp in DBFS
csv_temp_local = "/tmp/gold_olist_csvs_local" # local driver folder
os.makedirs(csv_temp_local, exist_ok=True)    # make sure local folder exists


In [0]:
#Copy all files from csv_temp to csv_temp_local
for file in dbutils.fs.ls(csv_temp):
    local_file_path = os.path.join(csv_temp_local, file.name.strip('/'))     
    dbutils.fs.cp(file.path, f"file:{local_file_path}")

In [0]:
os.listdir(csv_temp_local)

In [0]:
#Remove files with csv.crc:
for file in os.listdir(csv_temp_local):
    if file.endswith('.csv.crc'):
        os.remove(os.path.join(csv_temp_local, file))

In [0]:
#List out files in csv_temp_local
os.listdir(csv_temp_local)

In [0]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
#Make zip path 
zip_folder = f"tmp/gold_olist_zip_files"
##Make zip path: 
zip_path=f'{zip_folder}/gold_olist_{timestamp}'

# Ensure local folder exists
os.makedirs(os.path.dirname(zip_folder), exist_ok=True)

#Create zip file
shutil.make_archive(zip_path, 'zip', csv_temp_local)



In [0]:
#Make dbfs file to get link download
dbfs_folder = "dbfs:/FileStore/gold_olist_zip_files"
dbutils.fs.mkdirs(dbfs_folder)  # make sure DBFS folder exists

zip_dbfs = f"{dbfs_folder}/{os.path.basename(zip_path)}.zip"

# Copy local ZIP to DBFS
dbutils.fs.cp(f"file:{os.path.abspath(zip_path)}.zip", zip_dbfs, True)

print(f'File available in {zip_dbfs}')

Then, the zip file is downloaded in this link: https://{workspace}.azuredatabricks.net/files/gold_olist_zip_files/gold_olist_{timstamp}.zip

In [0]:
dbutils.fs.ls(csv_temp)

In [0]:
df=spark.read.csv("dbfs:/dbfs/tmp/gold_olist_csvs/dim_customer.csv" ,header=True, inferSchema=True)

In [0]:
df.summary().show()

In [0]:
spark.sql('select count(*) from gold_olist.dim_customer').show()